# Grover's algorithm: why AES-128 is "weakened", not "broken"

pqc-scan labels AES-128 as **WEAKENED** and AES-256 as **PQ_SAFE**. This
notebook shows why, with a tiny quantum simulator written in plain Python
(no libraries needed).

**The problem.** A key search is "find the one key out of N that works".
A classical computer needs about N/2 tries on average. Grover's algorithm
needs only about (π/4)·√N steps — a *square-root* speed-up.

In [ ]:
import math


def grover(n_qubits: int, marked: int) -> tuple[int, float]:
    """Simulate Grover search over 2**n_qubits items. Returns (steps, success probability)."""
    N = 2**n_qubits
    amp = [1 / math.sqrt(N)] * N  # equal superposition: every key at once
    steps = round(math.pi / 4 * math.sqrt(N))
    for _ in range(steps):
        amp[marked] = -amp[marked]  # 1) oracle: flip the sign of the right key
        mean = sum(amp) / N  # 2) diffusion: reflect every amplitude
        amp = [2 * mean - a for a in amp]  #    around the average
    return steps, amp[marked] ** 2  # probability of measuring the right key


for n in (4, 8, 12):
    steps, p = grover(n, marked=3)
    print(
        f"{2**n:>5} keys: classical ~{2**n // 2:>4} tries, Grover {steps:>3} steps, success {p:.1%}"
    )

Each step makes the right answer a little "louder" (larger amplitude) and the
wrong ones quieter. After about (π/4)·√N steps, measuring gives the right key
with probability close to 100%.

In [ ]:
steps, p = grover(10, marked=123)
assert p > 0.99 and steps == 25  # 1024 keys found in 25 steps

## What this means for real keys

√N for an n-bit key is 2^(n/2): Grover halves the *effective* key length.

| Key | Classical work | With Grover | pqc-scan |
| --- | --- | --- | --- |
| AES-128 | 2^128 | ~2^64 | WEAKENED |
| AES-192 | 2^192 | ~2^96 | WEAKENED |
| AES-256 | 2^256 | ~2^128 | PQ_SAFE |

2^128 is still far beyond any computer, so AES-256 stays safe. Grover also
cannot be run in parallel efficiently, which makes the real attack even
harder. Hash functions are affected in a similar way, which is why SHA-384/512
are preferred for long-term use.

In [ ]:
for bits in (128, 192, 256):
    print(f"AES-{bits}: effective security against Grover ≈ 2^{bits // 2}")